# SmartDs Spherical Shell Mass Flux (Starter)

This notebook loads the 3D BATSRUS sample file, samples a spherical shell at `R=10`, computes the radial mass flux on that shell, and plots it as a 2D map.

It reuses the **library** for shell sampling and mass-loss integration (no VTK/PyVista, no custom resampling).

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.analysis.shells import (
    infer_body_radius_m,
    integrate_shell_scalar,
    resolve_batsrus_density_si,
    resolve_batsrus_vector_xyz_si,
    sample_spherical_shells,
)
from starwinds_analysis.analysis.mass_loss import mass_loss_vs_radius
from starwinds_analysis.recipes.spherical import spherical_vector_components

plt.rcParams['figure.dpi'] = 120


## Load the 3D Sample File

Defaults to `sample_data/3d__var_3_n00060000.plt`.

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'sample_data').exists():
    REPO_ROOT = Path('/Users/dagfev/Documents/starwinds/starwinds-analysis')

DATA_FILE = REPO_ROOT / 'sample_data' / '3d__var_3_n00060000.plt'
if not DATA_FILE.exists():
    candidates = sorted((REPO_ROOT / 'sample_data').glob('3d*00060000*.plt'))
    if not candidates:
        available = sorted(p.name for p in (REPO_ROOT / 'sample_data').glob('*.plt'))
        raise FileNotFoundError('3D n00060000 sample not found. Available .plt files: ' + ', '.join(available))
    DATA_FILE = candidates[0]

print('Using:', DATA_FILE)
sds = SmartDs.from_file(str(DATA_FILE))
print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))


## Sample a Spherical Shell at `R = 10` and Compute Mass Flux

The shell sampling uses the library grid sampler (for plotting on a regular `theta/phi` grid).

In [ ]:
R_SHELL = 10.0  # in BATSRUS coordinate units (typically body radii)
N_POLAR = 48
N_AZIMUTH = 96

# body_radius_m = infer_body_radius_m(sds)
body_radius_m = 696_000_000  # in meters
print(f"Body radius [m]: {body_radius_m:.6e}")

rho_name, rho_scale = resolve_batsrus_density_si(sds)
(ux_name, uy_name, uz_name), u_scale = resolve_batsrus_vector_xyz_si(sds, "U")
print("Density field:", rho_name, "-> SI scale", rho_scale)
print("Velocity fields:", (ux_name, uy_name, uz_name), "-> SI scale", u_scale)

shell = sample_spherical_shells(
    sds,
    [R_SHELL],
    fields=(rho_name, ux_name, uy_name, uz_name),
    n_polar=N_POLAR,
    n_azimuth=N_AZIMUTH,
    method="nearest",
    length_unit_to_m=body_radius_m,
)

rho = rho_scale * shell.fields[rho_name]
ux = u_scale * shell.fields[ux_name]
uy = u_scale * shell.fields[uy_name]
uz = u_scale * shell.fields[uz_name]

u_r, _u_theta, _u_phi = spherical_vector_components(ux, uy, uz, shell.x, shell.y, shell.z)
mass_flux = rho * u_r  # kg / m^2 / s

mass_flux_shell = mass_flux[0]
mass_flux_finite = mass_flux_shell[np.isfinite(mass_flux_shell)]
print("Mass-flux finite cells:", mass_flux_finite.size, "/", mass_flux_shell.size)
print("Mass-flux range [kg/m^2/s]:", float(np.nanmin(mass_flux_finite)), float(np.nanmax(mass_flux_finite)))


In [ ]:
from matplotlib.ticker import MultipleLocator

def style_shell_lonlat_axes(ax, *, title):
    """Reusable shell lon/lat axis dressing for full-sphere maps."""
    ax.set_xlabel("Longitude [deg]")
    ax.set_ylabel("Latitude [deg]")
    ax.set_title(title)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    # Use tick spacings that are factors of 90 so full-sphere coverage is obvious.
    ax.xaxis.set_major_locator(MultipleLocator(90))
    ax.xaxis.set_minor_locator(MultipleLocator(30))
    ax.yaxis.set_major_locator(MultipleLocator(45))
    ax.yaxis.set_minor_locator(MultipleLocator(15))
    ax.tick_params(which="major", length=5)
    ax.tick_params(which="minor", length=3)
    ax.grid(which="major", alpha=0.15, linewidth=0.5)
    return ax


In [ ]:
# 2D shell plot in longitude/latitude using the native theta/phi sampling grid
# We assume the wind mass flux is mostly outward (>0). Non-positive values are shown with the colormap under-color.
lon_deg = np.degrees(shell.phi)
lat_deg = 90.0 - np.degrees(shell.theta)

MASS_FLUX_NAME = "Mass flux"
MASS_FLUX_UNIT = "kg/m^2/s"

finite = np.isfinite(mass_flux_shell)
positive = finite & (mass_flux_shell > 0.0)
if not positive.any():
    raise ValueError("No positive mass flux values found on the shell; cannot use log scaling.")

vmin = float(np.nanpercentile(mass_flux_shell[positive], 5.0))
vmax = float(np.nanpercentile(mass_flux_shell[positive], 99.0))
vmin = max(vmin, np.finfo(float).tiny)
if vmax <= vmin:
    vmax = vmin * 10.0

# Map non-positive values to a positive number below vmin so they use the under-color.
under_value = max(np.nextafter(vmin, 0.0), vmin * 1e-3)
mass_flux_plot = np.array(mass_flux_shell, dtype=float, copy=True)
mass_flux_plot[finite & (mass_flux_plot <= 0.0)] = under_value

cmap = plt.get_cmap("viridis").copy()
cmap.set_under("magenta")
norm = LogNorm(vmin=vmin, vmax=vmax)

fig, ax = plt.subplots(figsize=(9, 4.5))
img = ax.pcolormesh(lon_deg, lat_deg, mass_flux_plot, shading="nearest", cmap=cmap, norm=norm)
style_shell_lonlat_axes(ax, title=f"{MASS_FLUX_NAME} on shell at R={R_SHELL:g}")
cbar = fig.colorbar(img, ax=ax, extend="min")
cbar.set_label(f"{MASS_FLUX_NAME} [{MASS_FLUX_UNIT}]")
n_nonpos = int(np.count_nonzero(finite & (mass_flux_shell <= 0.0)))
print(f"Non-positive cells shown with under-color: {n_nonpos}")
plt.show()


## Mass-Loss Rate at `R = 10` (Library Function)

This uses `mass_loss_vs_radius(...)` with **Fibonacci-sphere sampling** for the shell integral.
For notebook responsiveness we use a lower integration resolution than the plotting shell.
(The runtime scales with the number of shell sample points.)

In [ ]:
# Optional cross-check: integrate the plotted grid-shell mass flux directly
mass_loss_grid, coverage_grid = integrate_shell_scalar(mass_flux, shell.area)

# Use a smaller Fibonacci shell for the integral so this cell stays responsive.
# Effective Fibonacci sample count is N_MASS_LOSS_POLAR * N_MASS_LOSS_AZIMUTH.
N_MASS_LOSS_POLAR = 16
N_MASS_LOSS_AZIMUTH = 32
print(f"Fibonacci integration points: {N_MASS_LOSS_POLAR * N_MASS_LOSS_AZIMUTH}")

profile = mass_loss_vs_radius(
    sds,
    [R_SHELL],
    n_polar=N_MASS_LOSS_POLAR,
    n_azimuth=N_MASS_LOSS_AZIMUTH,
    sampling="fibonacci",
    method="nearest",
)

dotm_lib = float(profile["mass_loss [kg/s]"][0])
cov_lib = float(profile["coverage [none]"][0])

print(f"Grid-shell integral (same plotted shell): {float(mass_loss_grid[0]):.6e} kg/s")
print(f"Grid-shell coverage             : {float(coverage_grid[0]):.6f}")
print(f"Library mass_loss_vs_radius     : {dotm_lib:.6e} kg/s")
print(f"Library coverage                : {cov_lib:.6f}")
